# Pareto front generation via the augmented ε-constraint method

This example traces the **energy-vs-comfort Pareto front** for the space-heater model from
`optimizer_example.ipynb` using `Optimizer.pareto_front` — no evolutionary algorithm involved.
Because the Twin4Build simulator is differentiable, every front point is an *exact
gradient-based NLP solve*.

**The two objectives:**

- **f₁ — heating energy**: mean space-heater power over the horizon (`min`).
- **f₂ — thermal discomfort**: mean of the residual `relu(T_setpoint − T_zone)` (`min`), i.e.
  the average underheating below the comfort setpoint in Kelvin. The residual is computed by a
  `FunctionSystem` — a generic helper component that applies a user-supplied torch
  transformation to named input signals, so *any* derived quantity can act as an objective or
  constraint.

**The method (AUGMECON):**

1. **Anchor solves**: minimize each objective alone → ideal/nadir estimates of f₂.
2. **ε sweep**: keep f₁ as the objective, demote f₂ to a hard constraint `f2_norm ≤ ε`, and
   sweep ε between the anchors. A small `δ·f2_norm` term guarantees *properly* Pareto-optimal
   points, and — unlike a weighted sum — the ε-constraint scheme recovers non-convex front
   regions.
3. **Batched prepass** (default on): all ε-subproblems are first solved approximately as ONE
   batched torch loss (`torch.func.vmap` over the composed rollout — one backward pass per
   iteration yields every copy's gradient), then the exact sequential SLSQP solves polish the
   batched solutions. This is exactly the batched workload shape where a GPU pays off: on a
   CUDA machine, add `model.to("cuda")` after `model.load()`.

The front answers a concrete engineering question: **how much heating energy does one Kelvin
of average comfort cost?** — the reported slope `-df₁/dε` is that marginal price at each point.


## 1. Build the model

Same physics as `optimizer_example.ipynb`: an RC thermal zone
(`BuildingSpaceThermalTorchSystem`) coupled to a hydronic radiator
(`SpaceHeaterTorchSystem`) whose water flow is the decision variable, driven by outdoor
temperature and supply-water schedules.

New here is the **`Discomfort` FunctionSystem**: it receives the heating-setpoint schedule
(21 °C during working hours, 18 °C setback) and the simulated zone temperature, and outputs
`relu(setpoint − T_zone)` — zero when the room is warm enough, the deficit in Kelvin when it
is not. Because its `forward` is a pure torch expression, it composes into the fast
optimization objective and the GPU-batched prepass exactly like any built-in component, and
gradients flow through it.


In [ ]:
# Install (uncomment on Colab)
# %pip install -q twin4build

import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dateutil import tz

import twin4build as tb

model = tb.Model(id="pareto_example")

building_space = tb.BuildingSpaceThermalTorchSystem(
    C_air=2e6, C_wall=1e7, R_out=0.005, R_in=0.005,
    f_wall=0, f_air=0, Q_occ_gain=100.0, CO2_occ_gain=0.004,
    CO2_start=400.0, infiltrationRate=0.0, airVolume=100.0,
    id="BuildingSpace",
)
space_heater = tb.SpaceHeaterTorchSystem(
    Q_flow_nominal_sh=2000.0, T_a_nominal_sh=60.0, T_b_nominal_sh=30.0,
    TAir_nominal_sh=21.0, thermalMassHeatCapacity=500000.0, nelements=3,
    id="SpaceHeater",
)

# Disturbances (fixed boundary conditions)
outdoor_temp = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 10.0,
        "ruleset_start_minute": [0] * 7, "ruleset_end_minute": [0] * 7,
        "ruleset_start_hour": [0, 6, 12, 18, 21, 23, 24],
        "ruleset_end_hour": [6, 12, 18, 21, 23, 24, 24],
        "ruleset_value": [5.0, 8.0, 15.0, 12.0, 8.0, 5.0, 5.0],
    },
    id="OutdoorTemperature",
)
zero = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 0.0}, id="Zero")
supply_air_temp = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 20.0}, id="SupplyAirTemperature")
supply_water_temp = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 60.0}, id="SupplyWaterTemperature")

# The decision variable: the heater's water flow (baseline: crude on/off profile).
mf = space_heater.Q_flow_nominal_sh / 4180 / (space_heater.T_a_nominal_sh - space_heater.T_b_nominal_sh)
waterflow_schedule = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 0,
        "ruleset_start_minute": [0, 0], "ruleset_end_minute": [0, 0],
        "ruleset_start_hour": [8, 19], "ruleset_end_hour": [16, 20],
        "ruleset_value": [mf, mf],
    },
    id="WaterflowSchedule",
)

# Comfort setpoint profile (21 degC working hours, 18 degC setback)
heating_setpoint = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 18.0,
        "ruleset_start_minute": [0] * 7, "ruleset_end_minute": [0] * 7,
        "ruleset_start_hour": [0, 8, 17, 0, 0, 0, 0],
        "ruleset_end_hour": [8, 16, 24, 0, 0, 0, 0],
        "ruleset_value": [18.0, 21.0, 18.0, 18.0, 18.0, 18.0, 18.0],
    },
    id="HeatingSetpoint",
)

# The second objective: thermal discomfort as a derived signal.
discomfort = tb.FunctionSystem(
    inputs=["setpoint", "measured"],
    fn=lambda d: torch.relu(d["setpoint"] - d["measured"]),
    id="Discomfort",
)

model.add_connection(zero, building_space, "scheduleValue", "numberOfPeople")
model.add_connection(outdoor_temp, building_space, "scheduleValue", "outdoorTemperature")
model.add_connection(zero, building_space, "scheduleValue", "globalIrradiation")
model.add_connection(zero, building_space, "scheduleValue", "supplyAirFlowRate")
model.add_connection(zero, building_space, "scheduleValue", "exhaustAirFlowRate")
model.add_connection(supply_air_temp, building_space, "scheduleValue", "supplyAirTemperature")
model.add_connection(supply_water_temp, space_heater, "scheduleValue", "supplyWaterTemperature")
model.add_connection(waterflow_schedule, space_heater, "scheduleValue", "waterFlowRate")
model.add_connection(building_space, space_heater, "indoorTemperature", "indoorTemperature")
model.add_connection(space_heater, building_space, "Power", "heatGain")
model.add_connection(heating_setpoint, discomfort, "scheduleValue", "setpoint")
model.add_connection(building_space, discomfort, "indoorTemperature", "measured")

model.load()

# GPU: uncomment on a CUDA machine -- the batched prepass is the part that benefits.
# model.to("cuda")

simulator = tb.Simulator(model)
optimizer = tb.Optimizer(simulator)

START = datetime.datetime(2024, 1, 4, tzinfo=tz.gettz("Europe/Copenhagen"))
END = START + datetime.timedelta(days=2)
STEP = 2400  # 40 min -> 72 decision variables


## 2. Trace the front

`objective1` is kept as the scalar objective; `objective2` is swept via the ε constraint.
Both use the familiar `(component, port, "min"|"max")` format from `Optimizer.optimize` —
the discomfort signal is just another output port. This takes a few minutes on a laptop CPU.


In [ ]:
result = optimizer.pareto_front(
    start_time=START,
    end_time=END,
    step_size=STEP,
    variables=[(waterflow_schedule, "scheduleValue", 0.0, mf)],
    objective1=(space_heater, "Power", "min"),      # mean heater power [W]
    objective2=(discomfort, "output", "min"),       # mean underheating [K]
    n_points=6,
    batched_prepass=True,
    options={"maxiter": 50},
)

df = pd.DataFrame({
    "eps": result.eps,
    "mean heater power [W]": result.f1,
    "mean discomfort [K]": result.f2,
    "slope -df1/deps": result.slope,
    "success": result.success,
    "iterations": result.nit,
    "pareto": result.pareto_mask,
})
df


## 3. The front

Each point is a full 2-day water-flow trajectory. The left end of the front (ε = 1) is the
energy anchor — no heating, maximum discomfort; the right end (ε = 0) drives the average
underheating to (near) zero at maximum energy. The slope annotations give the local exchange
rate: extra normalized energy per unit of comfort. Note the increasing steepness toward zero
discomfort — squeezing out the last tenths of a Kelvin is the expensive part, which is exactly
the information a single-solution optimization cannot give you.

`success=False` on a point simply means SLSQP hit `maxiter` before formally declaring
convergence — the point still satisfies its ε constraint (increase `maxiter` for tighter
polishing).


In [ ]:
ax = result.plot()
for i in range(1, len(result.eps) - 1):
    ax.annotate(
        f"{result.slope[i]:.2f}",
        (result.f2[i], result.f1[i]),
        textcoords="offset points", xytext=(8, -4), fontsize=8, alpha=0.8,
    )
ax.set_xlabel("mean discomfort [K]")
ax.set_ylabel("mean heater power [W]")
ax.set_title("Energy vs comfort Pareto front (AUGMECON)")
plt.tight_layout()
plt.show()


## 4. Inspect one solution

`result.apply(i)` writes point `i`'s decision trajectories back into the model and
re-simulates. Compare a comfort-leaning point against the baseline plot in
`optimizer_example.ipynb`: the optimized flow pre-heats before the 8h setpoint step and rides
the setpoint instead of overshooting.


In [ ]:
i = len(result.eps) - 2  # a comfort-leaning compromise
result.apply(i)

hours = np.arange(int((END - START).total_seconds() / STEP)) * STEP / 3600
T = building_space.output["indoorTemperature"].history(i_s=0).detach().cpu().numpy().ravel()
sp = heating_setpoint.output["scheduleValue"].history(i_s=0).detach().cpu().numpy().ravel()
P = space_heater.output["Power"].history(i_s=0).detach().cpu().numpy().ravel()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(hours, T, color="tab:blue", label="zone temperature")
ax1.plot(hours, sp, color="tab:gray", ls="--", label="heating setpoint")
ax1.fill_between(hours, T, sp, where=sp > T, color="tab:red", alpha=0.3, label="discomfort")
ax1.set_ylabel("temperature [degC]")
ax1.legend(loc="lower right")
ax2.step(hours, P, where="post", color="tab:red")
ax2.set_ylabel("heater power [W]")
ax2.set_xlabel("hours")
ax1.set_title(
    f"Pareto point {i}: mean power {result.f1[i]:.0f} W, "
    f"mean discomfort {result.f2[i]:.3f} K"
)
plt.tight_layout()
plt.show()


## Notes and limits

- **`FunctionSystem` is the general mechanism**: any torch expression of model signals can be
  an objective or constraint — e.g. weighted discomfort `relu(sp − T)**2`, an energy cost
  signal `power * price` (see also `ScalarProductSystem`), or CO₂ excess above a limit. The
  callable must be a pure, differentiable torch expression; note that a Python lambda cannot
  be serialized into the semantic model.
- **A varying baseline matters**: decision-variable ports normalize with their cached history
  min/max, so give the decision schedule a non-constant baseline profile (like the on/off
  profile here) — a constant baseline makes the normalization degenerate and the solver stalls.
- **GPU batching**: the sequential SLSQP polish is CPU-bound (scipy), but the batched prepass
  evaluates all ε-copies as one tensor program — with `model.to("cuda")` that is the
  batched-kernel workload where the GPU pays off (see the GPU scaling benchmark notebook).
- **Bi-objective only**: ε-grids scale poorly beyond ~3 objectives (use NBI-style methods
  there); a uniform ε grid gives non-uniform point spacing on steep front segments; each point
  is locally optimal (inherited NLP non-convexity).
- The reported slope is a finite-difference estimate; exact constraint multipliers arrive with
  the IPOPT-based batched finisher (follow-up work).
